In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
sys.path.append('..')

import utilities.functions as functions


from utilities.functions import (
    summary,
    gerar_resumo_decis,
    resumo_coorte_ativa
 )
from utilities.graficos import (
  boxplot_meses
 
)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from utilities.outliers import(
    outlier_method,
    mark_outliers_iqr_zscore_mad
)
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 


In [ ]:
df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")
df.head()

In [ ]:
id_both_monht=df[df['order_created_month']==12]['customer_id'].unique()
df = df[df['customer_id'].isin(id_both_monht)].reset_index(drop=True)


In [ ]:
df_stats_mes = df.groupby(['order_created_month', 'is_target'])['order_total_amount'].agg(
    Média=('mean'),
    Mediana=('median'),
    Mínimo=('min'),
    Máximo=('max'),
    Desvio_Padrão=('std')
).round(2)
df_stats_mes['cv'] = (df_stats_mes['Desvio_Padrão'] / df_stats_mes['Média']).round(3)
df_stats_mes

In [ ]:
boxplot_meses(df,var_cat='order_created_month',var_cont='order_total_amount')

In [ ]:
outlier_method(df,var='order_total_amount')

In [ ]:
df = mark_outliers_iqr_zscore_mad(df)


In [ ]:
total_amount = df.groupby(['order_created_month','is_target','outlier_iqr','outlier_zscore','outlier_mad']).agg(
    total_pedidos=('unique_order_hash', 'nunique')
).round(2)
total_amount

Consentracao de outliers no target --- isso ira influenciar roi

In [ ]:
df_outliers = df[
    df['outlier_iqr'] & 
    df['outlier_zscore'] & 
    df['outlier_mad']][['customer_id','order_created_month','is_target','order_total_amount','num_pedidos_hist','num_pedidos_mes']]
print(len(df_outliers))
df_outliers.describe()


In [ ]:
boxplot_meses(df_outliers,var_cat='order_created_month',var_cont='order_total_amount')

Analisando outliers excluindo o amount 138750

In [ ]:
df_outliers[df_outliers['order_total_amount']>138750]['customer_id'].unique()
#361e229dbc1b985e1aacb3e70384782a05d77ad6db53e7e511fe2147ee09a890

In [ ]:
boxplot_meses(df_outliers[df_outliers['order_total_amount']<138750],var_cat='order_created_month',var_cont='order_total_amount')

In [ ]:
df_outliers[df_outliers['order_total_amount']<138750].describe()

Analisando outliers excluindo o amount 8358

In [ ]:
df_outliers[(df_outliers['order_total_amount']>8358) &(df_outliers['order_total_amount']<138750)]['customer_id'].unique()
#dbae4016df6932dc7a0c8ba1b50fe79b670483d4dbc1a473157c6e8a0945d1d5'

In [ ]:
boxplot_meses(df_outliers[df_outliers['order_total_amount']<8358],var_cat='order_created_month',var_cont='order_total_amount')

In [ ]:
df_outliers[df_outliers['order_total_amount']<8358].describe()

In [ ]:
p99 = df['order_total_amount'].quantile(0.99)

acima_p99 = df[
    df['order_total_amount'] > p99
]

acima_p99[['customer_id','unique_order_hash','order_created_month','order_total_amount','outlier_iqr','outlier_zscore','outlier_mad']]

acima_p99['customer_id'].nunique()

In [ ]:
id_p99=acima_p99['customer_id'].unique()

In [ ]:
df['id_p99'] = df['customer_id'].isin(id_p99).astype(int)
df[df['id_p99']==1][['customer_id','unique_order_hash','order_created_month','order_total_amount','day','outlier_iqr','outlier_zscore','outlier_mad','id_p99']].head(10)

In [ ]:
stats_p99_por_cliente = (
    df[df['id_p99'] == 1]
    .groupby(['customer_id','order_created_month','is_target'])['order_total_amount']
    .agg(
        total_transacoes='count',
        minimo='min',
        maximo='max',
        media='mean'
    )
    .reset_index()   
    .sort_values(by=['customer_id','total_transacoes'], ascending=[False,False])
)

stats_p99_por_cliente.head(10)




In [ ]:
p10 = df['order_total_amount'].quantile(0.10)

abaixo_p10 = df[
    df['order_total_amount'] < p10
]

abaixo_p10[['customer_id','unique_order_hash','order_created_month','order_total_amount','outlier_iqr','outlier_zscore','outlier_mad']]

abaixo_p10['customer_id'].unique()


In [ ]:
id_p10=abaixo_p10['customer_id'].unique()
df['id_p10'] = df['customer_id'].isin(id_p10).astype(int)
df[df['id_p10']==1][['customer_id','unique_order_hash','order_created_month','order_total_amount','day','outlier_iqr','outlier_zscore','outlier_mad','id_p99']].head(10)
stats_p10_por_cliente = (
    df[df['id_p10'] == 1]
    .groupby(['customer_id','order_created_month','is_target'])['order_total_amount']
    .agg(
        total_transacoes='count',
        minimo='min',
        maximo='max',
        media='mean'
    )
    .reset_index()   
    .sort_values(by=['customer_id','total_transacoes'], ascending=[False,False])
)
stats_p10_por_cliente.head(10)

In [ ]:
df.to_parquet(BASE_PATH / "gold" / "df_publico.parquet", index=False)

# Calcular LTV histórico E total de pedidos
historico_cliente = df[df['customer_id'] == '002ca070253de5dd983f577e0156dae30cfe981198bce5c7815755748343594c']

ltv_historico = historico_cliente['order_total_amount'].sum()
total_pedidos = len(historico_cliente)
tempo_atividade = (historico_cliente['order_created_at'].max() - historico_cliente['order_created_at'].min()).days

# Calcular métricas derivadas
ticket_medio = ltv_historico / total_pedidos if total_pedidos > 0 else 0

# LTV mensal projetado
if tempo_atividade > 0:
    ltv_mensal = (ltv_historico / tempo_atividade) * 30
    ltv_anual_projetado = ltv_mensal * 12
    frequencia_mensal = (total_pedidos / tempo_atividade) * 30
else:
    ltv_anual_projetado = ltv_historico * 4
    frequencia_mensal = total_pedidos

print("📊 ANÁLISE COMPLETA DO CLIENTE:")
print(f"💰 LTV Histórico: R$ {ltv_historico:.2f}")
print(f"🛒 Total de Pedidos: {total_pedidos}")
print(f"📈 LTV Anual Projetado: R$ {ltv_anual_projetado:.2f}")
print(f"💵 Ticket Médio: R$ {ticket_medio:.2f}")
print(f"📅 Frequência Mensal: {frequencia_mensal:.1f} pedidos/mês")
print(f"⏰ Tempo de Atividade: {tempo_atividade} dias")

# Análise adicional
if total_pedidos > 0:
    print(f"\n🎯 MÉTRICAS AVANÇADAS:")
    print(f"• Valor por Dia: R$ {ltv_historico/tempo_atividade:.2f}" if tempo_atividade > 0 else "• Valor por Dia: N/A")
    print(f"• Pedidos por Dia: {total_pedidos/tempo_atividade:.3f}" if tempo_atividade > 0 else "• Pedidos por Dia: N/A")
    
    # Classificação do cliente
    if ltv_anual_projetado > 5000:
        classificacao = "💎 CLIENTE PREMIUM"
    elif ltv_anual_projetado > 2000:
        classificacao = "🥇 CLIENTE OURO" 
    elif ltv_anual_projetado > 1000:
        classificacao = "🥈 CLIENTE PRATA"
    else:
        classificacao = "👤 CLIENTE BÁSICO"
    
    print(f"• Classificação: {classificacao}")
